# 实验5：后门检测技术（Neural Cleanse）

本实验用于课堂展示后门检测流程：基于 Neural Cleanse 思路，对已训练的 BadNets 模型逆向搜索潜在触发器，定位可疑目标类，并结合 STRIP++ 样本级检测结果理解不同检测视角。

它承接实验4：实验4负责制造后门，本实验负责在模型和样本层面解释“为什么目标类 0 可疑”。


## 一、使用说明

本 notebook 可独立运行：如果实验4已经产生现场 checkpoint，会优先加载最新的 `demo_last.ckpt`；如果没有，则自动回退到包内官方 checkpoint。正式 Neural Cleanse 结论读取服务器完整 43 类反演结果。课堂展示时建议按以下顺序讲解：

- 先固定检测对象和 checkpoint 来源。
- 区分 sample-level 的 STRIP++ 和 model-level 的 Neural Cleanse。
- 先看正式检测总览，再看 43 类反演和 MAD。
- 最后回扣实验4的 clean accuracy/ASR，说明检测结论不等同于修复。

运行后重点观察四类信息：

- `suspected_target_class`：Neural Cleanse 认为最可疑的目标类。
- `target_label_0_detected`：检测是否命中实验4预设的目标类 0。
- `mask_norm_class_0` 与 `second_smallest_mask_norm`：目标类所需触发器是否明显更小。
- `mad_anomaly_index`：MAD 异常分数，超过阈值通常表示存在可疑后门。


## 二、环境与依赖初始化

本实验直接复用实验4已经安装的依赖与 CANN 内核。确认右上角为 **Python 3.11.4 (CANN)**，运行下面的简短检查后继续实验；尚未完成配置时，请先运行实验4的环境初始化部分。

<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
table th,
table td {
  text-align: left !important;
}
</style>


In [ ]:
import os
import sys

if not os.environ.get("ASCEND_TOOLKIT_HOME"):
    raise RuntimeError(
        "请先完成实验4的环境初始化，并选择 Python 3.11.4 (CANN) 内核。"
    )

from tbe.common import utils

print(f"CANN ready: {os.environ['ASCEND_TOOLKIT_HOME']}")
print(f"Python: {sys.executable}")
print("TBE utils OK")
import os
import sys
import json
import warnings
from pathlib import Path

os.environ.setdefault("GLOG_v", "3")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np

# 实验5默认独立运行；如需更快的现场演示，可在启动前设置 ONSITE_DEMO_PROFILE=fast。
RUN_PROFILE = os.environ.get("ONSITE_DEMO_PROFILE", "cloud_live").strip().lower()
if RUN_PROFILE not in {"fast", "enhanced", "cloud_live"}:
    raise RuntimeError(f"Unsupported RUN_PROFILE: {RUN_PROFILE}")
DEMO_MODE = RUN_PROFILE
PREFERRED_DEVICE = "Ascend"

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break

from demo_lib.paths import (
    ensure_dir,
    find_checkpoint,
    get_detection_paths,
    load_demo_config,
    load_final_summary,
    make_run_timestamp,
    resolve_demo_mode_settings,
    resolve_project_root,
)
from demo_lib.runtime import configure_mindspore_device
from demo_lib.subset import create_demo_subset
from demo_lib.inference import load_model_from_checkpoint
import importlib
import demo_lib.detection as detection_module
import demo_lib.interactive_demo as interactive_demo_module

# 允许在同一个 Notebook kernel 中重新运行时加载本地 demo_lib 修复。
importlib.reload(detection_module)
importlib.reload(interactive_demo_module)

from demo_lib.interactive_demo import (
    collect_attack_images,
    run_detection_widget_demo_if_available,
    run_free_detection_test,
    show_attack_image_gallery,
    show_strip_light_detection_result,
)
from demo_lib.detection import (
    build_attack_detection_overview_rows,
    load_formal_detection_bundle,
    save_strip_light_result,
)
from demo_lib.detection_visualization import (
    plot_formal_strip_metrics,
    plot_neural_cleanse_anomaly_comparison,
    plot_neural_cleanse_mask_norms,
    show_attack_detection_overview,
    show_formal_detection_table,
    show_neural_cleanse_summary,
)

try:
    import ipywidgets as widgets
    from IPython.display import Markdown, display
    WIDGETS_AVAILABLE = True
    WIDGET_IMPORT_ERROR = ""
except Exception as e:
    from IPython.display import Markdown, display
    WIDGETS_AVAILABLE = False
    WIDGET_IMPORT_ERROR = repr(e)

paths = resolve_project_root()
PROJECT_ROOT = paths["PROJECT_ROOT"]
CONFIG = load_demo_config()
MODE_SETTINGS = resolve_demo_mode_settings(CONFIG, DEMO_MODE)
TARGET_LABEL = int(CONFIG["target_label"])
SEED = int(CONFIG["seed"])
MS_MODE = str(CONFIG.get("ms_mode", "PYNATIVE"))
TRAIN_PER_CLASS = int(MODE_SETTINGS["train_per_class"])
TEST_PER_CLASS = int(MODE_SETTINGS["test_per_class"])
BATCH_SIZE = int(MODE_SETTINGS["batch_size"])
DEMO_RUN_ROOT = ensure_dir(paths["DEMO_RUNS_ROOT"] / f"notebook_exp5_{make_run_timestamp()}")
NOTEBOOK_DEMO_SUBSET_ROOT = ensure_dir(DEMO_RUN_ROOT / "demo_subset")
paths["DEMO_SUBSET_ROOT"] = NOTEBOOK_DEMO_SUBSET_ROOT
DETECTION_OUTPUT_DIR = ensure_dir(DEMO_RUN_ROOT / "detection")

summary = load_final_summary()
DETECTION_PATHS = get_detection_paths()
DETECTION_CONFIG = CONFIG["detection"]
DETECTION_BATCH_SIZE = int(DETECTION_CONFIG.get("strip_light_batch_size", 1))


def display_table(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


def widget_fallback_markdown():
    return Markdown(
        "当前环境没有可用的 ipywidgets，下面切换到普通 cell 兜底展示。"
        "安装 `ipywidgets` 和 `jupyterlab_widgets` 后重开 notebook 可恢复交互。"
    )


def latest_demo_checkpoint(stage_dir_name):
    candidates = sorted(
        paths["DEMO_RUNS_ROOT"].glob(f"notebook*/{stage_dir_name}/demo_last.ckpt"),
        key=lambda p: p.stat().st_mtime if p.exists() else 0,
        reverse=True,
    )
    return candidates[0] if candidates else None


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"run profile = {RUN_PROFILE}")
print(f"target label = {TARGET_LABEL}")
print(f"batch size = {BATCH_SIZE}")
print(f"ipywidgets available = {WIDGETS_AVAILABLE}")
print(f"demo run output = {DEMO_RUN_ROOT}")
print(f"strip light batch size = {DETECTION_BATCH_SIZE}")
print(f"formal detection summary = {DETECTION_PATHS['DETECTION_SUMMARY_JSON']}")


## 三、读取测试数据与样本浏览

先确认 GTSRB 测试集可用，并构建用于现场 light STRIP++ 演示的小规模 demo subset。这个 subset 用于样本级交互展示；Neural Cleanse 正式结论不依赖它，而是读取服务器完整 43 类反演结果。


In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

ensure_exp5_context(globals(), require_subset=True)

data_gallery_items = collect_attack_images(
    project_root=PROJECT_ROOT,
    source="data_test",
    max_images=12,
    skip_target_label=False,
    target_label=TARGET_LABEL,
)
print(f"展示样本数：{len(data_gallery_items)}")
display(show_attack_image_gallery(data_gallery_items, max_images=12, cols=6))
plt.close("all")

subset_manifest_path = paths["DEMO_SUBSET_ROOT"] / "subset_manifest.json"
if subset_manifest_path.exists():
    subset_manifest = json.loads(subset_manifest_path.read_text(encoding="utf-8"))
else:
    subset_manifest = create_demo_subset(
        train_dir=paths["TRAIN_DIR"],
        test_dir=paths["TEST_DIR"],
        output_dir=paths["DEMO_SUBSET_ROOT"],
        train_per_class=TRAIN_PER_CLASS,
        test_per_class=TEST_PER_CLASS,
        seed=SEED,
    )
print(f"test total = {subset_manifest['test_total']}")
print(f"subset_manifest.json = {subset_manifest_path}")


#### 图像注释：检测阶段测试样本浏览

这张宫格用于确认实验5读取到的仍是 GTSRB 测试图像，并且样本标签与实验4的攻击目标保持同一套口径。每个小图展示真实交通标志类别，后续 STRIP++ 会从这些候选样本中选择非目标类图片进行触发样本检测。当前输出显示 12 张测试样本，并打印 demo subset 的 test total 为 129，说明现场轻量检测和正式服务器结果使用的是可追溯的数据入口。


#### 讲解：检测使用哪些数据

light STRIP++ fallback 使用 demo subset 做交互演示；Neural Cleanse 是模型级检测，核心是对每个候选类别逆向搜索一个最小触发器。课堂上可以先用样本级检测建立直观现象，再用完整 43 类反演记录给出模型级结论。


## 四、模型来源与 checkpoint 检查

检测对象是实验4训练出的 BadNets 模型。默认只检查 checkpoint 文件路径，不加载 MindSpore 模型；正式 STRIP++ / Neural Cleanse 结果读取不需要实时推理，因此不会占用 Ascend 显存。这个设计适合教学现场：先保证结论可展示，再按需开启实时演示。


In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

# 默认只检查 checkpoint 文件，不加载 MindSpore 模型，避免 Ascend 显存池不足。
RUN_LOAD_DETECTION_MODELS = False

ensure_exp5_context(globals(), require_subset=False, require_mindspore=False, require_models=False)
from demo_lib.paths import find_checkpoint

checkpoint_rows = []
for trigger_type in ("square", "checkerboard"):
    stage_cfg = CONFIG[trigger_type]
    ckpt_path = find_checkpoint(stage_cfg["experiment_name"])
    checkpoint_rows.append({
        "trigger_type": trigger_type,
        "experiment_name": stage_cfg["experiment_name"],
        "checkpoint_exists": ckpt_path.exists(),
        "checkpoint_path": str(ckpt_path),
        "model_loaded": False,
        "note": "默认不加载模型；正式 Neural Cleanse 结果不需要实时 Ascend 推理。",
    })

display_table(checkpoint_rows)
print(f"strip light batch size = {DETECTION_BATCH_SIZE}")
print("RUN_LOAD_DETECTION_MODELS = False，因此本格不会占用 Ascend 显存。")

if RUN_LOAD_DETECTION_MODELS:
    ensure_exp5_context(globals(), require_subset=False, require_models=True)
    square_detection_bundle = DETECTION_MODEL_BUNDLES["square"]
    checkerboard_detection_bundle = DETECTION_MODEL_BUNDLES["checkerboard"]
    display_table([
        {
            "trigger_type": "square",
            "model_source": square_detection_bundle["model_source"],
            "checkpoint_path": square_detection_bundle["checkpoint_path"],
            "note": square_detection_bundle["model_reason"],
        },
        {
            "trigger_type": "checkerboard",
            "model_source": checkerboard_detection_bundle["model_source"],
            "checkpoint_path": checkerboard_detection_bundle["checkpoint_path"],
            "note": checkerboard_detection_bundle["model_reason"],
        },
    ])
    print(f"MindSpore version = {ms.__version__}")
    print(f"device target = {ACTUAL_DEVICE}")


#### 讲解：为什么检测要固定模型来源

后门检测必须明确检测的是哪一个模型。若模型来自现场训练，结果体现当前演示；若来自官方 checkpoint，结果体现服务器完整实验。表格中的 `model_source` 和 `checkpoint_path` 用来记录这个来源。这是实验4/5统一口径的关键：攻击指标、检测指标和 checkpoint 应该指向同一模型来源。


## 五、实验任务与检测目标

本实验的检测问题是：在不知道触发器具体图案的情况下，判断实验4模型是否存在异常目标类，并解释异常来自哪里。课堂上可以把它拆成两层：

- 样本级检测：给定一张输入，判断它是否可能被 Trigger 激活。
- 模型级检测：给定一个模型，搜索哪个类别只需要异常小的 Trigger 就能诱导大量样本。

后续 STRIP++ 单元对应第一层，Neural Cleanse 反演与 MAD 异常分数对应第二层。


## 六、light STRIP++ 样本级检测演示

这一节用于现场交互演示“带 Trigger 的样本是否异常”。它不是本实验的 Neural Cleanse 正式结论；为避免 Ascend 显存池不足影响一键运行，fallback 单元默认跳过，需要演示时再手动开启。


### 6.1 fallback 检测

In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

# 实时运行 Ascend light STRIP++；本实验要求不使用 CPU fallback。
DETECTION_FALLBACK_TRIGGER_TYPE = "square"  # 可改为 "checkerboard"
RUN_LIGHT_STRIP_LIVE_ASCEND = True          # 现场实验要求真实运行 Ascend；不要改成 CPU fallback
RUN_LIGHT_STRIP_ISOLATED_PROCESS = True     # 推荐保持 True：隔离 MindSpore Ascend 显存池
DETECTION_LIVE_DEVICE_TARGET = "Ascend"
DETECTION_LIVE_MS_MODE = "PYNATIVE"           # 实时单样本演示优先降低图编译/缓存压力
DETECTION_LIVE_RANDOM_SEED = None
DETECTION_FALLBACK_SOURCE = "data_test"
DETECTION_FALLBACK_RANDOM_PICK = True
DETECTION_FALLBACK_IMAGE_INDEX = 1
DETECTION_FALLBACK_SKIP_TARGET_LABEL = True
RELEASE_NOTEBOOK_MODELS_BEFORE_LIVE_ASCEND = True

trigger = str(DETECTION_FALLBACK_TRIGGER_TYPE).strip().lower()
if trigger not in {"square", "checkerboard"}:
    raise ValueError("DETECTION_FALLBACK_TRIGGER_TYPE 只能是 'square' 或 'checkerboard'。")
experiment = "square_main" if trigger == "square" else "checkerboard_main"

fallback_detection_result = None
ensure_exp5_context(globals(), require_subset=False, require_mindspore=False, require_models=False)

if not RUN_LIGHT_STRIP_LIVE_ASCEND:
    formal_detection_bundle = load_formal_detection_bundle(summary)
    formal_rows = [row for row in formal_detection_bundle["formal_rows"] if row["experiment"] == experiment]

    display(Markdown(
        "当前单元使用非实时 fallback：只读取正式 STRIP++ 结果和已有缓存，"
        "不会加载模型，也不会占用 Ascend 显存。只有手动开启实时 light STRIP++ 推理时，"
        "才可能因 Ascend 显存池不足失败；该实时分支已默认关闭。"
    ))
    display_table(formal_rows)

    cached_paths = sorted(
        paths["DEMO_RUNS_ROOT"].glob(f"notebook*/detection/{trigger}_strip_light_result.notebook.json"),
        key=lambda path: path.stat().st_mtime if path.exists() else 0,
        reverse=True,
    )
    if cached_paths:
        fallback_detection_result_path = cached_paths[0]
        fallback_detection_result = json.loads(fallback_detection_result_path.read_text(encoding="utf-8"))
        cached_row = {
            "trigger_type": fallback_detection_result.get("trigger_type"),
            "candidate_image": fallback_detection_result.get("candidate_image_name") or fallback_detection_result.get("candidate_image"),
            "true_label": fallback_detection_result.get("true_label"),
            "suspicious_score": fallback_detection_result.get("suspicious_score"),
            "demo_threshold": fallback_detection_result.get("demo_threshold"),
            "detected_triggered": fallback_detection_result.get("detected_triggered"),
            "model_source": fallback_detection_result.get("model_source"),
            "cache_path": str(fallback_detection_result_path),
        }
        print("cached light STRIP++ result:")
        display_table([cached_row])
    else:
        print(f"没有找到 {trigger} 的缓存 light STRIP++ 结果；上面的正式 STRIP++ 指标仍然可用。")

    try:
        display(show_formal_detection_table(formal_rows))
        display(plot_formal_strip_metrics(formal_rows))
        plt.close("all")
    except ModuleNotFoundError as exc:
        if getattr(exc, "name", "") != "matplotlib":
            raise
        print("matplotlib 不可用，已只显示表格。")
else:
    display(Markdown(
        "正在尝试实时 Ascend light STRIP++ 推理。默认使用独立 Python 子进程，"
        "避免继承当前 notebook kernel 中训练/评估留下的 MindSpore Ascend 显存池。"
    ))
    DETECTION_FALLBACK_K = int(MODE_SETTINGS["strip_light_k"])

    if RUN_LIGHT_STRIP_ISOLATED_PROCESS:
        if RELEASE_NOTEBOOK_MODELS_BEFORE_LIVE_ASCEND:
            import gc

            for name in (
                "square_model",
                "checkerboard_model",
                "square_live_model",
                "checkerboard_live_model",
                "DETECTION_MODEL_BUNDLES",
            ):
                if name in globals():
                    globals()[name] = None
            gc.collect()
            try:
                import mindspore as _ms

                empty_cache = getattr(getattr(_ms, "hal", None), "empty_cache", None)
                if callable(empty_cache):
                    empty_cache()
            except Exception:
                pass

        import subprocess
        from IPython.display import Image as IPyImage
        from demo_lib.detection_visualization import (
            plot_strip_light_target_probability,
            show_strip_light_score_card,
            show_strip_light_summary_table,
        )

        live_script = PROJECT_ROOT / "src" / "onsite_demo" / "scripts" / "run_notebook_live_detection.py"
        live_cmd = [
            sys.executable,
            str(live_script),
            "--trigger-type", trigger,
            "--image-source", DETECTION_FALLBACK_SOURCE,
            "--random-pick", str(bool(DETECTION_FALLBACK_RANDOM_PICK)).lower(),
            "--image-index", str(int(DETECTION_FALLBACK_IMAGE_INDEX)),
            "--skip-target-label", str(bool(DETECTION_FALLBACK_SKIP_TARGET_LABEL)).lower(),
            "--k", str(int(DETECTION_FALLBACK_K)),
            "--seed", str(int(SEED)),
            "--random-seed", "" if DETECTION_LIVE_RANDOM_SEED is None else str(int(DETECTION_LIVE_RANDOM_SEED)),
            "--run-profile", str(RUN_PROFILE),
            "--device-target", str(DETECTION_LIVE_DEVICE_TARGET),
            "--ms-mode", str(DETECTION_LIVE_MS_MODE).upper(),
            "--subset-root", str(paths["DEMO_SUBSET_ROOT"]),
            "--output-dir", str(DETECTION_OUTPUT_DIR),
        ]
        completed = subprocess.run(
            live_cmd,
            cwd=str(PROJECT_ROOT),
            text=True,
            capture_output=True,
            check=False,
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.returncode != 0:
            if completed.stderr:
                print(completed.stderr[-4000:])
            message = (completed.stdout or "") + "\n" + (completed.stderr or "")
            ascend_memory_error_markers = (
                "Memory pool not enough",
                "Malloc device memory failed",
                "Memory_Allocation_Failure",
                "Available memory is insufficient",
                "may be other processes occupying this card",
                "rtMalloc failed",
                "free size:",
            )
            if any(marker in message for marker in ascend_memory_error_markers):
                display(Markdown(
                    "独立子进程仍然遇到 Ascend 显存不足，且本实验配置为不使用 CPU fallback。"
                    "请先关闭或重启其它占用 Ascend 的 notebook kernel / Python 进程，"
                    "确认 NPU 至少有几十 MB 可用显存后再重跑本单元。"
                ))
                print("Ascend live demo unavailable even in isolated process: Ascend memory is occupied")
            else:
                raise RuntimeError(f"实时检测子进程失败，returncode={completed.returncode}")
        else:
            fallback_detection_result_path = DETECTION_OUTPUT_DIR / f"{trigger}_strip_light_result.notebook.live.json"
            fallback_detection_result = json.loads(fallback_detection_result_path.read_text(encoding="utf-8"))
            print(f"fallback trigger_type = {trigger}")
            print(f"fallback result path = {fallback_detection_result_path}")
            print(f"model_source = {fallback_detection_result['model_source']}")
            print(f"threshold_source = {fallback_detection_result['threshold_source']}")
            print(f"batch_size = {fallback_detection_result.get('batch_size')}")
            print(f"device target = {fallback_detection_result.get('device_target')}")
            display(show_strip_light_score_card(fallback_detection_result))
            display(show_strip_light_summary_table(fallback_detection_result))
            display(plot_strip_light_target_probability(fallback_detection_result))
            preview_path = Path(str(fallback_detection_result.get("preview_png_path", "")))
            if preview_path.exists():
                display(IPyImage(filename=str(preview_path)))
            plt.close("all")
    else:
        ensure_exp5_context(
            globals(),
            require_subset=True,
            require_models=True,
            model_trigger_type=trigger,
        )
        DETECTION_MODEL_BUNDLES = {trigger: DETECTION_MODEL_BUNDLES[trigger]}
        fallback_detection_result = run_free_detection_test(
            project_root=PROJECT_ROOT,
            subset_test_dir=paths["DEMO_SUBSET_ROOT"] / "test",
            detection_model_bundles=DETECTION_MODEL_BUNDLES,
            detection_config=DETECTION_CONFIG,
            image_source=DETECTION_FALLBACK_SOURCE,
            random_pick=DETECTION_FALLBACK_RANDOM_PICK,
            image_index=DETECTION_FALLBACK_IMAGE_INDEX,
            trigger_type=trigger,
            skip_target_label=DETECTION_FALLBACK_SKIP_TARGET_LABEL,
            random_seed=DETECTION_LIVE_RANDOM_SEED,
            target_label=TARGET_LABEL,
            k=int(DETECTION_FALLBACK_K),
            seed=SEED,
        )
        fallback_detection_result_path = save_strip_light_result(
            fallback_detection_result,
            DETECTION_OUTPUT_DIR / f"{trigger}_strip_light_result.notebook.json",
        )
        print(f"fallback trigger_type = {trigger}")
        print(f"fallback result path = {fallback_detection_result_path}")
        print(f"model_source = {fallback_detection_result['model_source']}")
        print(f"threshold_source = {fallback_detection_result['threshold_source']}")
        print(f"batch_size = {fallback_detection_result.get('batch_size')}")
        print(f"device target = {ACTUAL_DEVICE}")
        print(f"MindSpore version = {ms.__version__}")
        show_strip_light_detection_result(fallback_detection_result)
        plt.close("all")


#### 图像注释：light STRIP++ 样本级检测结果

这一组图从样本级角度判断“当前输入是否像被触发”。候选对比图展示 clean 样本与 triggered 样本的视觉差异；扰动/blend 网格展示把候选图与多张参考图混合后的结果；目标类概率曲线观察每次扰动后 `P(target=0)` 是否仍然异常稳定；分数卡和汇总图给出 suspicious score、threshold 与 detected_triggered。当前缓存输出中 square 样本的 suspicious score 约为 0.9981，高于阈值约 0.6592，因此 `detected_triggered=True`；这说明该样本在多次扰动下仍强烈指向目标类，符合触发样本特征。


### 6.2 widgets 检测


In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

# widgets 检测需要加载模型并执行 Ascend 推理；本实验要求不使用 CPU fallback。
RUN_DETECTION_WIDGETS = False

ensure_exp5_context(globals(), require_subset=True, require_mindspore=False, require_models=False)

display(Markdown("widgets 检测会加载模型并执行 Ascend 实时样本级演示；本实验不使用 CPU fallback。"))

if not RUN_DETECTION_WIDGETS:
    display(Markdown(
        "当前未启动 widgets 实时检测。正式 STRIP++ / Neural Cleanse 结果从下一节读取，"
        "不需要加载模型，也不占用 Ascend 显存。"
    ))
elif WIDGETS_AVAILABLE:
    ensure_exp5_context(globals(), require_subset=True, require_models=True)
    detection_widget_result = run_detection_widget_demo_if_available(
        project_root=PROJECT_ROOT,
        subset_test_dir=paths["DEMO_SUBSET_ROOT"] / "test",
        detection_model_bundles=DETECTION_MODEL_BUNDLES,
        detection_config=DETECTION_CONFIG,
        target_label=TARGET_LABEL,
        default_image_source="data_test",
        default_trigger_type="square",
        default_k=int(MODE_SETTINGS["strip_light_k"]),
        seed=SEED,
        enable_widgets=True,
        result_output_dir=DETECTION_OUTPUT_DIR,
    )
    if isinstance(detection_widget_result, tuple) and detection_widget_result[0] is False:
        WIDGETS_AVAILABLE = False
        WIDGET_IMPORT_ERROR = detection_widget_result[1]

    if not WIDGETS_AVAILABLE:
        display(widget_fallback_markdown())
        if WIDGET_IMPORT_ERROR:
            print(WIDGET_IMPORT_ERROR)
    else:
        display(Markdown("Trigger 下拉框可切换 square / checkerboard。"))
else:
    display(widget_fallback_markdown())
    if WIDGET_IMPORT_ERROR:
        print(WIDGET_IMPORT_ERROR)


#### 讲解：STRIP++ 与 Neural Cleanse 的区别

STRIP++ 关注单个输入样本是否像被触发；Neural Cleanse 关注整个模型是否存在某个异常小的触发器。前者是 sample-level detection，后者是 model-level detection。


## 七、正式检测结果总览

读取打包好的服务器检测结果。表格同时列出攻击指标、STRIP++ 样本级检测指标和 Neural Cleanse 模型级检测指标。


In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

ensure_exp5_context(globals(), require_subset=False, require_mindspore=False, require_models=False)

formal_detection_bundle = load_formal_detection_bundle(summary)
formal_detection_rows = formal_detection_bundle["formal_rows"]

try:
    display(show_formal_detection_table(formal_detection_rows))
    display(plot_formal_strip_metrics(formal_detection_rows))
    plt.close("all")
except ModuleNotFoundError as exc:
    if getattr(exc, "name", "") != "matplotlib":
        raise
    display_table(formal_detection_rows)
    print("matplotlib 不可用，已退回表格输出。")


#### 图像注释：正式检测指标总览

第一张图是服务器正式结果表，把攻击指标、STRIP++ 指标和 Neural Cleanse 指标放在同一行中，便于检查“攻击是否成功”和“检测是否命中”是否同时成立。第二张图把 STRIP++ 的 detection rate、ROC-AUC、PR-AUC 和 FPR 画成柱状对比：detection rate/AUC 越高越好，FPR 越低越好。当前正式结果中 square 的 STRIP++ detection rate 为 0.8933、FPR 为 0.0067、ROC-AUC 为 0.9550、PR-AUC 为 0.9643，说明样本级检测能较好地区分触发样本与干净样本。


## 八、Neural Cleanse 反演与 MAD 异常检测（可单独运行）

Neural Cleanse 对每个候选目标类都反演一个触发器 mask：如果某个类别只需要异常小的 mask 就能让大量样本被预测成该类，这个类别就可能是后门目标类。随后使用 MAD（Median Absolute Deviation）而不是均值/方差计算异常分数，以降低极端值对判断的影响。


In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

ensure_exp5_context(globals(), require_subset=False, require_mindspore=False, require_models=False)

formal_detection_bundle = globals().get("formal_detection_bundle") or load_formal_detection_bundle(summary)
formal_detection_rows = formal_detection_bundle["formal_rows"]
nc_summary = formal_detection_bundle["detection_summary"]["neural_cleanse_full_43_class"]

nc_rows = []
for experiment in ("square_main", "checkerboard_main"):
    item = nc_summary[experiment]
    nc_rows.append({
        "experiment": experiment,
        "completed_class_count": int(item["completed_class_count"]),
        "suspected_target_class": int(item["suspected_target_class"]),
        "target_label_0_detected": bool(item["target_label_0_detected"]),
        "mask_norm_class_0": round(float(item["mask_norm_class_0"]), 4),
        "second_smallest_mask_norm": round(float(item["second_smallest_mask_norm"]), 4),
        "median_mask_norm": round(float(item["median_mask_norm"]), 4),
        "mad_anomaly_index": round(float(item["mad_anomaly_index"]), 4),
        "anomaly_threshold": round(float(item["anomaly_threshold"]), 4),
        "reversed_success_rate": round(float(item["reversed_success_rate"]), 4),
    })

display_table(nc_rows)
print(f"Neural Cleanse acceptance_pass = {bool(nc_summary['acceptance_pass'])}")

try:
    nc_summary_fig = show_neural_cleanse_summary(formal_detection_rows)
    display(nc_summary_fig)
    plt.close(nc_summary_fig)

    nc_anomaly_fig = plot_neural_cleanse_anomaly_comparison(nc_summary)
    display(nc_anomaly_fig)
    plt.close(nc_anomaly_fig)

    nc_mask_fig = plot_neural_cleanse_mask_norms(nc_summary)
    display(nc_mask_fig)
    plt.close(nc_mask_fig)
except ModuleNotFoundError as exc:
    if getattr(exc, "name", "") != "matplotlib":
        raise
    print("matplotlib 不可用，Neural Cleanse 图形已跳过，表格结果已显示。")


#### 图像注释：Neural Cleanse 反演与 MAD 异常

这一组图从模型级角度解释后门目标类。表格先列出 43 类反演是否完成、最可疑类别、目标类 0 的 mask L1、第二小 L1、中位数 L1、MAD anomaly index 和阈值；后续图则把这些异常程度可视化。当前结果中 square_main 的 `mask_norm_class_0=242.2166`，明显低于第二小值 499.6801 和中位数 669.8555，MAD anomaly index 为 4.5259；checkerboard_main 的 `mask_norm_class_0=116.1734`，低于第二小值 379.8344 和中位数 593.5593，MAD anomaly index 为 6.3364。两者都超过阈值 2.0，且 suspected target class 都是 0，因此 Neural Cleanse 成功定位了实验4设置的后门目标类。


#### 讲解：为什么 L1 越小越可疑

对正常类别来说，把任意输入强行推到该类别通常需要较大、分散的扰动；对后门目标类来说，模型已经学会了固定 Trigger，因此逆向搜索能找到更小的 mask。`mask_norm_class_0` 明显小于第二小值和中位数时，说明目标类 0 具有异常触发器特征。


#### 讲解：用 MAD 做异常检测

MAD 使用中位数作为中心，统计每个类别 mask L1 与中位数的偏离程度。相比均值/方差，MAD 对少数异常类别更稳健，适合 Neural Cleanse 这种“只有一个或少数类别异常”的检测场景。


## 九、攻击与检测最终总览（可单独运行）


In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "notebook_bootstrap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break
else:
    raise FileNotFoundError("Could not locate src/onsite_demo/demo_lib/notebook_bootstrap.py")

from demo_lib.notebook_bootstrap import ensure_exp5_context

ensure_exp5_context(globals(), require_subset=False, require_mindspore=False, require_models=False)

formal_detection_bundle = globals().get("formal_detection_bundle") or load_formal_detection_bundle(summary)
attack_detection_overview_rows = build_attack_detection_overview_rows(
    summary,
    formal_detection_bundle["detection_summary"],
    formal_detection_bundle["detection_table"],
)
try:
    display(show_attack_detection_overview(attack_detection_overview_rows))
    plt.close("all")
except ModuleNotFoundError as exc:
    if getattr(exc, "name", "") != "matplotlib":
        raise
    display_table(attack_detection_overview_rows)
    print("matplotlib 不可用，已退回表格输出。")


#### 图像注释：攻击与检测闭环总览

这张总览图把每个实验的攻击结果、STRIP++ 样本级检测结果、Neural Cleanse 模型级检测结果和最终 `detection_passed` 放在一起。它的作用不是提供新的算法指标，而是把实验4和实验5串成闭环：先证明模型存在有效后门，再证明检测流程能在样本层面发现可疑输入、在模型层面定位可疑目标类。若图中两个实验都显示 `detection_passed=True`，就说明 square 与 checkerboard 两条攻击链都被检测流程覆盖。


## 十、实验5结果解读
- 检测命中：当前正式结果显示，square_main 和 checkerboard_main 的 Neural Cleanse 都完成了 43 类反演，最可疑类别均为 `target_label=0`，且 `target_label_0_detected=True`。
- 异常强度：square 的 MAD anomaly index 约为 4.5259，checkerboard 约为 6.3364，均高于阈值 2.0；反演触发器验证成功率均为 1.0。
- 结论边界：这说明 Neural Cleanse 能正确定位本实验中植入的后门目标类，但不等价于模型已经被修复或防御成功。


## 十一、MindSpore 与 CANN 的作用

MindSpore 在本实验中主要负责可选模型加载和实时推理；CANN/Ascend 在开启 STRIP++ widgets 或实时检测时提供推理算子执行。默认流程直接读取服务器完整检测结果，因此可以在显存紧张时仍稳定展示 Neural Cleanse 结论；需要现场实时推理时再确认 MindSpore/CANN 版本、device target 和可用显存。


## 结论

实验5完成了后门检测流程：加载实验4模型，展示可选样本级检测，并基于服务器完整 43 类 Neural Cleanse 反演结果，用 MAD 异常分数定位后门目标类。课堂上可把它总结为：检测能定位 `target_label=0` 的异常触发器特征，但检测结论不等于模型修复；若要进入防御/修复，需要额外的模型清洗或再训练流程。
